In [1]:
# Necessary Libraries
import pandas as pd
import re

## 1. Dataset Preparation

In this section, we load and pre-process all of the datasets. 

### 1.1 Gini Coefficient

In [2]:
# Load the datasets
df_gini = pd.read_csv('economic-inequality-gini-index.csv')

In [3]:
# Remove leading & trailing whitespace + replace multiple spaces within names to a single underscore
df_gini.columns = df_gini.columns.str.strip().str.replace(r'\s+', '_', regex=True)

# Duplicate
# Automatically detect categorical/object columns
cat_cols = df_gini.select_dtypes(include=['object', 'category']).columns
# Strip leading and trailing spaces from all categorical columns
df_gini[cat_cols] = df_gini[cat_cols].apply(lambda x: x.str.strip())

# Check dataset structures
print("Dataset Structures: ")
display(df_gini.info())

# First 5 rows
print("\nFirst 5 rows: ")
display(df_gini.head())

# Last 5 rows
print("\nLast 5 rows: ")
display(df_gini.tail())

# Check the descriptive statistics of the datasete
print('\nCategorical Variables:')
display(df_gini.describe(include = 'object'))

print('\nNumerical Variables: ')
display(df_gini.describe(include = 'number'))

# Check missing values 
print("\nMissing values count for each column: ")
display(df_gini.isnull().sum())  

print("\nMissing values percentage: ")
display(df_gini.isna().mean() * 100)   

num_duplicates = df_gini.duplicated().sum()
print(f"\nTotal duplicate rows: {num_duplicates}")

Dataset Structures: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2285 entries, 0 to 2284
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Entity              2285 non-null   object 
 1   Code                2152 non-null   object 
 2   Year                2285 non-null   int64  
 3   Gini_coefficient    2285 non-null   float64
 4   990179-annotations  0 non-null      float64
dtypes: float64(2), int64(1), object(2)
memory usage: 89.4+ KB


None


First 5 rows: 


,Entity,Code,Year,Gini_coefficient,990179-annotations
0,Albania,ALB,1996,0.270103,NaN
1,Albania,ALB,2002,0.317390,NaN
2,Albania,ALB,2005,0.305957,NaN
3,Albania,ALB,2008,0.299847,NaN
4,Albania,ALB,2012,0.289605,NaN



Last 5 rows: 


...


Missing values count for each column: 


Entity                   0
Code                   133
Year                     0
Gini_coefficient         0
990179-annotations    2285
dtype: int64


Missing values percentage: 


Entity                  0.000000
Code                    5.820569
Year                    0.000000
Gini_coefficient        0.000000
990179-annotations    100.000000
dtype: float64


Total duplicate rows: 0


In [19]:
df_gini.shape

(170, 3)

#### Drop Uncessary Columns 

In [4]:
df_gini.drop(columns=["990179-annotations"], inplace=True)
#df_gini.drop(columns=["990179-annotations", "Code"], inplace=True)

#### Columns Renaming

In [5]:
df_gini.rename(columns={
    'Entity': 'country',
    'Year': 'year_gini',
    'Code': 'country_code',
    'Gini_coefficient': 'gini'
}, inplace=True)

#### Filter for Year 2022 or the Most Recent Year Before 2022

In [6]:
# Filter only rows with year ≤ 2022
df_gini = df_gini[df_gini['year_gini'] <= 2022]

# For each country, get its latest year
latest_year_per_country = df_gini.groupby('country')['year_gini'].max().reset_index()

# Merge back to get full rows matching latest year for each country
df_gini = df_gini.merge(latest_year_per_country, on=['country', 'year_gini'], how='inner')

# Drop the year column 
df_gini.drop(columns=["year_gini"], inplace=True)

#### Handle Missing Value in Code

In [7]:
#df_gini[df_gini['code'].isna()].head()
df_gini.head(10)

,country,country_code,gini
0,Albania,ALB,0.294196
1,Algeria,DZA,0.276157
2,Angola,AGO,0.512640
3,Argentina (urban),NaN,0.407365
4,Armenia,ARM,0.279453
5,Australia,AUS,0.343326
6,Austria,AUT,0.307016
7,Azerbaijan,AZE,0.265549
8,Bangladesh,BGD,0.333732
9,Belarus,BLR,0.243834


#### Standardize Country Names

Check unique country names

In [8]:
df_gini['country'].unique()

array(['Albania', 'Algeria', 'Angola', 'Argentina (urban)', 'Armenia',
       'Australia', 'Austria', 'Azerbaijan', 'Bangladesh', 'Belarus',
       'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia',
       'Bolivia (urban)', 'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'Bulgaria', 'Burkina Faso', 'Burundi', 'Cameroon', 'Canada',
       'Cape Verde', 'Central African Republic', 'Chad', 'Chile', 'China',
       'China (rural)', 'China (urban)', 'Colombia', 'Colombia (urban)',
       'Comoros', 'Congo', 'Costa Rica', "Cote d'Ivoire", 'Croatia',
       'Cyprus', 'Czechia', 'Democratic Republic of Congo', 'Denmark',
       'Djibouti', 'Dominican Republic', 'East Timor', 'Ecuador',
       'Ecuador (urban)', 'Egypt', 'El Salvador', 'Estonia', 'Eswatini',
       'Ethiopia', 'Ethiopia (rural)', 'Fiji', 'Finland', 'France',
       'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece',
       'Grenada', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana',
       'Haiti', 'Honduras', '

In [9]:
df_gini[df_gini['country'].isin(['China', 'China (urban)', 'China (rural)'])].head(30)

,country,country_code,gini
28,China,CHN,0.357438
29,China (rural),NaN,0.304040
30,China (urban),NaN,0.356526


In [10]:
def clean_country_names(df, country_column='country'):

    """
    Cleans and filters country names in a dataset based on the following rules:
    
    1. If country-wide data exists (like 'China'), keep it and remove region-specific entries.
    2. If only regional data exists (like 'Argentina (urban)' and 'Argentina (rural)'), keep '(urban)' if available.
    3. If only one regional data exists, keep it and rename it to the base country (remove region info).

    Parameters:
    - df (pd.DataFrame): Your input dataframe
    - country_column (str): Column name containing country names (default='country')

    Returns:
    - pd.DataFrame: Cleaned dataframe with standardized country names
    """
    
    df = df.copy()

    # Remove "(country)" as part of some country naming string
    df[country_column] = df[country_column].str.replace(r'\(country\)', '', regex=True).str.strip()

    # Helper column to store country data
    df['base_country'] = df[country_column].str.replace(r'\s*\(.*\)', '', regex=True).str.strip()

    def pick_preferred(group):
        base_country = group.name
        if group[country_column].isin([base_country]).any():
            picked = group[group[country_column] == base_country]
        else:
            urban = group[group[country_column].str.contains(r'\(urban\)', regex=True)]
            picked = urban if not urban.empty else group.iloc[[0]]

        # Add base_country back manually
        picked = picked.assign(base_country=base_country)
        return picked

    # Apply filtering logic
    cleaned_df = df.groupby('base_country', group_keys=False).apply(pick_preferred, include_groups=False)

    # Final clean country name: set to base_country
    cleaned_df[country_column] = cleaned_df['base_country']

    # Drop helper columns
    cleaned_df.drop(columns=['base_country'], inplace=True)

    return cleaned_df.reset_index(drop=True)


In [11]:
df_gini = clean_country_names(df_gini, country_column='country')

In [12]:
df_gini[df_gini["country"].isin(['Micronesia'])].head()

,country,country_code,gini
97,Micronesia,FSM,0.400524


In [13]:
df_gini.head(10)

,country,country_code,gini
0,Albania,ALB,0.294196
1,Algeria,DZA,0.276157
2,Angola,AGO,0.512640
3,Argentina,NaN,0.407365
4,Armenia,ARM,0.279453
5,Australia,AUS,0.343326
6,Austria,AUT,0.307016
7,Azerbaijan,AZE,0.265549
8,Bangladesh,BGD,0.333732
9,Belarus,BLR,0.243834


In [14]:
df_gini.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       170 non-null    object 
 1   country_code  169 non-null    object 
 2   gini          170 non-null    float64
dtypes: float64(1), object(2)
memory usage: 4.1+ KB


In [15]:
df_gini[df_gini['country_code'].isna()].head()

,country,country_code,gini
3,Argentina,NaN,0.407365


In [16]:
# Fill missing 'code' where 'country' is 'ARG'
df_gini.loc[(df_gini['country'] == 'Argentina') & (df_gini['country_code'].isna()), 'country_code'] = 'ARG'

In [17]:
df_gini.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       170 non-null    object 
 1   country_code  170 non-null    object 
 2   gini          170 non-null    float64
dtypes: float64(1), object(2)
memory usage: 4.1+ KB


In [18]:
df_gini.to_csv('gini_procecssed.csv', index=False)